In [19]:
from pydantic import BaseModel
class PatientData(BaseModel):
    name: str
    age: int
    weight: float  # Added weight field to the PatientData model
def add_patient_data(patient: PatientData):
    # Here you can add logic to store the patient data in a database or perform other operations
    print(f"Name: {patient.name}, Age: {patient.age}, Weight: {patient.weight}")
patient_data = {"name": "John Doe", "age": 30, "weight": 70.5}
patient = PatientData(**patient_data)
# ** means unpacking the dictionary into keyword arguments for the PatientData constructor.
def update_patient_data(patient: PatientData):
    # Here you can add logic to update the patient data in a database or perform other operations
    print(f"Updating patient data for {patient.name}, Age: {patient.age}, Weight: {patient.weight}")  
add_patient_data(patient)

Name: John Doe, Age: 30, Weight: 70.5


In [20]:
patient_2_data={"name": "Jane Smith", "age": 25, "weight": 65.0 }
patient_2 = PatientData(**patient_2_data)

update_patient_data(patient_2)

Updating patient data for Jane Smith, Age: 25, Weight: 65.0


In [21]:
class PatientData(BaseModel):
    name: str
    age: int
    weight: float  
    married: bool
    allergies: list[str]
    contact_info: dict[str, str]  
patient_3_data = { 
    "name": "Alice Johnson",
    "age": 40,
    "weight": 68.0,
    "married": True,
    "allergies": ["peanuts", "shellfish"],
    "contact_info": {"email": "nigga@gmail.com", "phone": "555-1234"}
}
patient_3 = PatientData(**patient_3_data)   

Required and Optional Fields

In [22]:
from typing import Optional


class PatientData(BaseModel):
    name: str
    age: int
    weight: float  
    married: bool=False
    allergies: Optional[list[str]]=None
    contact_info: dict[str, str]  

Data Validation

In [23]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, ValidationError
from typing import Optional, Annotated

class PatientData(BaseModel):
    name: str = Annotated[str,Field(max_length=50,title="Patient's full name", description="The full name of the patient, limited to 50 characters.", example="John Doe")]
    age: int=Field(gt=0, lt=120)
    linkedin_url: AnyUrl
    weight: float
    email: EmailStr
    married: bool = False
    allergies: Optional[list[str]] = None
    contact_info: dict[str, str]

bad_patient_data = {
    "name": "Bob " * 20,
    "age": -50,
    "linkedin_url": "https://www.linkedin.com/in/bobbrown",
    "weight": 80.0,
    "email": "nisd@gmail",
    "married": False,
    "allergies": ["pollen"],
    "contact_info": {"email": "fden@gmailc,.om", "phone": "555-5678"}
}

try:
    PatientData(**bad_patient_data)
except ValidationError as e:
    print(e)

good_patient_data = {
    "name": "Bob Brown",
    "age": 50,
    "linkedin_url": "https://www.linkedin.com/in/bobbrown",
    "weight": 80.0,
    "email": "bob.brown@example.com",
    "married": False,
    "allergies": ["pollen"],
    "contact_info": {"email": "bob.brown@example.com", "phone": "555-5678"}
}

patient_4 = PatientData(**good_patient_data)
print(patient_4)

C:\Users\cyber\AppData\Local\Temp\ipykernel_22488\4148184331.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  name: str = Annotated[str,Field(max_length=50,title="Patient's full name", description="The full name of the patient, limited to 50 characters.", example="John Doe")]


2 validation errors for PatientData
age
  Input should be greater than 0 [type=greater_than, input_value=-50, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
email
  value is not a valid email address: The part after the @-sign is not valid. It should have a period. [type=value_error, input_value='nisd@gmail', input_type=str]
name='Bob Brown' age=50 linkedin_url=AnyUrl('https://www.linkedin.com/in/bobbrown') weight=80.0 email='bob.brown@example.com' married=False allergies=['pollen'] contact_info={'email': 'bob.brown@example.com', 'phone': '555-5678'}


In [24]:
from pydantic import field_validator
class PatientData(BaseModel):
    name: str
    age: int
    email: EmailStr
    weight: float  
    married: bool
    allergies: list[str]
    contact_info: dict[str, str]  

    @field_validator("email")
    @classmethod
    def email_validator(cls, value):
        valid_domains = ['hdfc.com','icici.com']
        domain_name=value.split('@')[-1]
        if domain_name not in valid_domains:
            raise ValueError(f"Email domain must be one of {valid_domains}")
        return value

    @field_validator("name")
    @classmethod
    def transform_name(cls, value): # cls and value here are parameters for the validator method. cls refers to the class itself, and value is the value of the field being validated.
        return value.upper()

patient_4_data = { 
    "name": "Alice Johnson",
    "age": 40,
    "email": "alice@hdfc.com",
    "weight": 68.0,
    "married": True,
    "allergies": ["peanuts", "shellfish"],
    "contact_info": {"email": "nigga@gmail.com", "phone": "555-1234"}
}
patient_4 = PatientData(**patient_4_data)   

In [25]:
print(patient_4.age)
print(patient_4.name)
print(patient_4.email)
print(patient_4.contact_info)
print(patient_4.married)

40
ALICE JOHNSON
alice@hdfc.com
{'email': 'nigga@gmail.com', 'phone': '555-1234'}
True


Model Validator

In [26]:
from pydantic import model_validator

class PatientData(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: list[str]
    contact_info: dict[str, str]

    @model_validator(mode="after") #mode here means that the validator will be called after the model is initialized, allowing you to access all fields of the model. before mode would be used if you want to validate fields before the model is fully initialized.
    def validate_emergency_contact(cls, model):
        if model.age>60 and 'emergency' not in model.contact_info:
            raise ValueError("Emergency contact is required for patients over 60 years old.")
        return model

C:\Users\cyber\AppData\Local\Temp\ipykernel_22488\1221726287.py:12: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.13/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode="after") #mode here means that the validator will be called after the model is initialized, allowing you to access all fields of the model. before mode would be used if you want to validate fields before the model is fully initialized.


Computed Fields

In [27]:
from pydantic import computed_field

class PatientData(BaseModel):
    name: str
    age: int
    email: EmailStr
    weight: float  
    married: bool
    allergies: list[str]
    contact_info: dict[str, str]

    @computed_field
    @property # this decorator is used to define a computed property in the PatientData class. The @property decorator makes it accessible like an attribute, so you can access it as patient.bmi instead of calling a method.
    def bmi(self) -> float:
        bmi_value=round(self.weight / (self.height ** 2), 2)
        return bmi_value
    

Nested Models

In [28]:
class Address(BaseModel):
    city: str
    state: str
    pin: str

class PatientData(BaseModel):
    name: str
    age: int
    weight: float  
    married: bool=False
    allergies: Optional[list[str]]=None
    address: Address  # Nested model for address

address_data = {
    "city": "New York",
    "state": "NY",
    "pin": "10001"
}
address1=Address(**address_data)
patient_5_data = {
    "name": "Charlie Brown",
    "age": 35,
    "weight": 75.0,
    "married": True,
    "allergies": ["dust"],
    "address": address1  # Passing the nested model as a dictionary
}
patient_5 = PatientData(**patient_5_data)

print(patient_5)
print(patient_5.address.city)
print(patient_5.address.state)
print(patient_5.address.pin)

name='Charlie Brown' age=35 weight=75.0 married=True allergies=['dust'] address=Address(city='New York', state='NY', pin='10001')
New York
NY
10001


Serialization

In [29]:
class Address(BaseModel):
    city: str
    state: str
    pin: str

class PatientData(BaseModel):
    name: str
    age: int
    weight: float  
    married: bool=False
    allergies: Optional[list[str]]=None
    address: Address  # Nested model for address

address_data = {
    "city": "New York",
    "state": "NY",
    "pin": "10001"
}
address1=Address(**address_data)
patient_5_data = {
    "name": "Charlie Brown",
    "age": 35,
    "weight": 75.0,
    "married": True,
    "allergies": ["dust"],
    "address": address1  # Passing the nested model as a dictionary
}
patient_5 = PatientData(**patient_5_data)

temp=patient_5.model_dump()
print(temp)
print(type(temp))
temp2=patient_5.model_dump_json()
print(temp2)
print(type(temp2))

{'name': 'Charlie Brown', 'age': 35, 'weight': 75.0, 'married': True, 'allergies': ['dust'], 'address': {'city': 'New York', 'state': 'NY', 'pin': '10001'}}
<class 'dict'>
{"name":"Charlie Brown","age":35,"weight":75.0,"married":true,"allergies":["dust"],"address":{"city":"New York","state":"NY","pin":"10001"}}
<class 'str'>
